# Embedding HF: Qwen3 8B FP16, two GPUs
Set ID_FROM / ID_TO (inclusive). Add Kaggle Secret HF_TOKEN with write access to OUTPUT_REPO. Enable Internet and two GPUs. Output is separate from crawl corpus. Resume uses content-addressed completed bundles on HF; an interrupted bundle is recomputed. Different sessions should use disjoint ID ranges. Model and source revisions must stay identical between sessions. Copy the printed revisions into the configuration after the first run. Never mix these 4096-dimensional vectors with the local 4B vectors.

Model reference: https://huggingface.co/Qwen/Qwen3-Embedding-8B . This is the strongest size in the Qwen3 embedding family, not a proven winner on this competition. FP16 weights are split across both GPUs (not two replicas). T4 normally has 16 GB per card; the notebook checks actual capacities. No teacher or training is involved.


In [ ]:
ID_FROM = 0
ID_TO = 50000
SOURCE_REPO = "AncientWings/AncientWings_R2AI_STAGE3_ds"
OUTPUT_REPO = SOURCE_REPO
SOURCE_REVISION = "main"  # pin to the printed commit for subsequent sessions
MODEL_REVISION = "main"   # pin to the printed commit for subsequent sessions
OUTPUT_FOLDER = "embeddings/qwen3_8b_v1"
MAX_HOURS = 10.5
DOCS_PER_BUNDLE = 64
BATCH_SIZE = 4


In [ ]:
%pip install -q "transformers>=4.51,<5" "accelerate>=1.6" huggingface_hub pyarrow numpy


In [ ]:
import base64, io, zipfile, sys
from pathlib import Path
PROJECT = Path('/kaggle/working/embedding_project')
PROJECT.mkdir(exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))).extractall(PROJECT)
sys.path.insert(0, str(PROJECT/'src'))


In [ ]:
import os, time, json, hashlib, shutil
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_HOME'] = '/kaggle/working/hf_home'
from huggingface_hub import HfApi, HfFileSystem, CommitOperationAdd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from transformers import AutoTokenizer, AutoModel
from vibiomir.corpus.hf_snapshot import _official_file, _rows, _remote_rows, _valid, LINKS_NAME, LINKS_SHA256
api=HfApi(token=os.environ['HF_TOKEN'])
assert 0 <= ID_FROM <= ID_TO
assert torch.cuda.device_count() >= 2, 'Enable two GPUs'
for i in range(2):
    print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_properties(i).total_memory/2**30, 'GiB')
source_sha=api.dataset_info(SOURCE_REPO, revision=SOURCE_REVISION).sha
model_sha=api.model_info('Qwen/Qwen3-Embedding-8B', revision=MODEL_REVISION).sha
config=dict(source=SOURCE_REPO, source_sha=source_sha, model='Qwen/Qwen3-Embedding-8B', model_sha=model_sha,
            dtype='float16', dim=4096, tokens=768, target=320, overlap=80, chunk_method="sentence_token_budget_v1", version=2)
fingerprint=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:24]
prefix=f'{OUTPUT_FOLDER}/{fingerprint}'
print('PIN SOURCE_REVISION =', source_sha, 'MODEL_REVISION =', model_sha)
print('HF output:', prefix)
existing=set(api.list_repo_files(OUTPUT_REPO, repo_type='dataset'))
fs=HfFileSystem(token=os.environ['HF_TOKEN'])
local=Path('/kaggle/working/embedding_processed'); local.mkdir(exist_ok=True)
links=_official_file(local, LINKS_NAME, LINKS_SHA256, local/'hf_cache', os.environ['HF_TOKEN'])
official={int(r['id']):r['url'] for r in _rows(links) if ID_FROM <= int(r['id']) <= ID_TO}
tok=AutoTokenizer.from_pretrained(config['model'],revision=model_sha,padding_side='left')
model=AutoModel.from_pretrained(config['model'],revision=model_sha,torch_dtype=torch.float16,
    attn_implementation='sdpa',device_map='balanced',
    max_memory={i:int(torch.cuda.get_device_properties(i).total_memory*0.82) for i in range(2)})
assert all(str(d) not in ('cpu','disk') for d in model.hf_device_map.values()), 'Model spilled outside GPUs'
model.eval(); model.config.use_cache=False
print(model.hf_device_map)


In [ ]:
from vibiomir.corpus.chunk import chunk_document

def chunks_for(doc):
    return chunk_document(doc, tokenizer=tok, target_tokens=320,
                          overlap_tokens=80, max_tokens=768)

def encode(rows):
    output=np.empty((len(rows),4096),dtype=np.float16)
    order=sorted(range(len(rows)),key=lambda i:len(rows[i]['text']))
    pos=0; batch_size=BATCH_SIZE
    while pos<len(order):
        ix=order[pos:pos+batch_size]
        texts=[(rows[i]['title']+'\n' if rows[i]['title'] else '')+rows[i]['text'] for i in ix]
        try:
            inputs=tok(texts,padding=True,truncation=False,return_tensors='pt').to(model.device)
            with torch.inference_mode():
                vectors=torch.nn.functional.normalize(model(**inputs).last_hidden_state[:,-1].float(),p=2,dim=1)
            values=vectors.cpu().numpy()
            assert np.isfinite(values).all()
            output[ix]=values
            del inputs,vectors
            pos+=len(ix)
        except torch.cuda.OutOfMemoryError:
            if batch_size==1: raise
            batch_size=max(1,batch_size//2)
            if 'inputs' in locals(): del inputs
            torch.cuda.empty_cache()
    return output

started=time.monotonic(); seen=set(); pending=[]; uploaded=skipped=0

def flush(docs):
    global uploaded,skipped
    if not docs: return
    docs=sorted(docs,key=lambda d:d['id'])
    signature=hashlib.sha256(json.dumps([(d['id'],d['url'],d['text'],d.get('title')) for d in docs],ensure_ascii=False).encode()).hexdigest()
    name=f'ids_{docs[0]["id"]}_{docs[-1]["id"]}_{signature[:20]}'
    remote=f'{prefix}/{name}'
    if remote+'/manifest.json' in existing:
        skipped+=len(docs); return
    rows=[c for d in docs for c in chunks_for(d)]
    if not rows: raise ValueError('Empty bundle')
    path=local/name; path.mkdir(exist_ok=True)
    pq.write_table(pa.Table.from_pylist(docs),path/'corpus.parquet',compression='zstd')
    pq.write_table(pa.Table.from_pylist(rows),path/'chunks.parquet',compression='zstd')
    np.save(path/'embeddings.npy',encode(rows))
    files={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in path.iterdir() if p.name!='manifest.json'}
    (path/'manifest.json').write_text(json.dumps(dict(config=config,documents=len(docs),chunks=len(rows),files=files,
        document_signature=signature,query_instruction='Instruct: Given a Vietnamese medical question, retrieve passages that answer it\nQuery: ')),encoding='utf-8')
    # One atomic commit publishes vectors, mapping, source text and completion marker together.
    for attempt in range(5):
        try:
            api.create_commit(OUTPUT_REPO,repo_type='dataset',commit_message=f'Embedding {name}',
                operations=[CommitOperationAdd(path_in_repo=remote+'/'+p.name,path_or_fileobj=str(p)) for p in path.iterdir()])
            break
        except Exception:
            if attempt==4: raise
            time.sleep(2**attempt)
    existing.add(remote+'/manifest.json'); uploaded+=len(docs)
    print(json.dumps(dict(uploaded_docs=uploaded,skipped_docs=skipped,last_id=docs[-1]['id'],
        hours=round((time.monotonic()-started)/3600,3),gpu_allocated_gib=[torch.cuda.memory_allocated(i)/2**30 for i in range(2)])),flush=True)
    shutil.rmtree(path)  # only this successfully committed bundle

files=sorted(f for f in api.list_repo_files(SOURCE_REPO,repo_type='dataset',revision=source_sha) if f.startswith('corpus/') and f.endswith('.parquet'))
stop=False
for filename in files:
    for row in _remote_rows(fs,SOURCE_REPO,source_sha,filename,[(ID_FROM,ID_TO)]):
        doc=_valid(row,official,'all')
        if doc is None or doc['id'] in seen: continue
        seen.add(doc['id']); pending.append(doc)
        if len(pending)==DOCS_PER_BUNDLE:
            flush(pending); pending=[]
            if time.monotonic()-started > MAX_HOURS*3600:
                stop=True; break
    if stop: break
if pending: flush(pending)
print('Session paused; run same settings again.' if stop else 'Selected ID range completed.',uploaded,skipped)


## Resume and parallel sessions
Keep ID range, source/model revisions and DOCS_PER_BUNDLE identical on restart: deterministic bundles are skipped using HF completion manifests. An interrupted uncommitted bundle is recalculated (at most 64 documents). Changing range boundaries or bundle size can create overlapping bundles: assign non-overlapping fixed ranges to teammates. This version rescans source shards to reconstruct bundles; it skips GPU work for completed bundles, not all network reads. It does not provide distributed locks for overlapping workers.

The first upload needs a writable existing OUTPUT_REPO. Do not put tokens in cells. Download each bundle's corpus, chunks, vectors and manifest together for downstream indexing, and verify SHA256. Local pipeline 4B cannot consume these 8B vectors unchanged: query encoding must use this exact 8B revision, pooling, 4096 dimensions and instruction. A merger/8B retrieval adapter is a separate downstream step; this notebook produces embedding artifacts only.
